In [0]:
# Create catalog and schema if they don't exist
spark.sql("CREATE CATALOG IF NOT EXISTS CICD_CATALOG")
spark.sql("CREATE SCHEMA IF NOT EXISTS CICD_CATALOG.INGEST_SCHEMA")

# Read billing usage data with date filter for performance (last 90 days)
df = spark.sql("""
    SELECT *
    FROM system.billing.usage
    WHERE usage_date >= current_date() - INTERVAL 90 DAYS
""")

# Drop and recreate table with correct schema from source
spark.sql("DROP TABLE IF EXISTS CICD_CATALOG.INGEST_SCHEMA.billing_usage")

# Create table using the DataFrame schema (ensures exact type match)
df.write.mode("overwrite").saveAsTable("CICD_CATALOG.INGEST_SCHEMA.billing_usage")

print(f"Successfully loaded {df.count()} rows into CICD_CATALOG.INGEST_SCHEMA.billing_usage")